In [37]:
%pip install duckdb pandas matplotlib seaborn plotly numpy openpyxl

Note: you may need to restart the kernel to use updated packages.


In [40]:
import pandas as pd
import duckdb

df_volunteers= pd.read_csv("../data/volunteers.csv")

df_interviews = pd.read_csv("../data/interviews.csv")

df_analytics = pd.read_csv("../data/analytics.csv")


## Connect with db + import csv files as datasets

In [41]:

con = duckdb.connect()

con.register("volunteers", df_volunteers)
con.register("interviews", df_interviews)
con.register("analytics", df_analytics)
con.execute("""
CREATE VIEW raw_volunteers AS
SELECT *
FROM read_csv_auto('../data/volunteers.csv');
""")

con.execute("""
CREATE VIEW raw_interviews AS
SELECT *
FROM read_csv_auto('../data/interviews.csv');
""")

con.execute("""
CREATE VIEW raw_analytics AS
SELECT *
FROM read_csv_auto('../data/analytics.csv');
""")



## Check datasets info

In [42]:
query = """
SELECT *
FROM raw_volunteers
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

                 Name                        Creation log  \
0          Mariam Ali  U+ HR Account Sep 11, 2025 1:30 PM   
1  Manusha Srikanthan     Jess Scott Sep 11, 2025 1:45 PM   
2          Harvi Shah     Jess Scott Sep 11, 2025 4:36 PM   
3      Dorsey Bangarh     Jess Scott Sep 11, 2025 5:37 PM   
4       Pushti Ladani     Jess Scott Sep 11, 2025 7:26 PM   
5        Purti Ladani     Jess Scott Sep 11, 2025 7:29 PM   
6        Harjot Singh     Jess Scott Sep 11, 2025 8:52 PM   
7       Rincy Nahomie     Jess Scott Sep 11, 2025 8:31 PM   
8          JeeHu Choi     Jess Scott Sep 11, 2025 8:55 PM   
9     Taranveer Arora     Jess Scott Sep 11, 2025 9:32 PM   

                             Email Address                        YRES Email  \
0              mariam.ali@yorkeducation.ca                               NaN   
1              manushasrikanthan@gmail.com                               NaN   
2                  harvishah2602@gmail.com       harvi.shah@yorkeducation.ca   
3       

In [43]:
query = """
SELECT *
FROM raw_interviews
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

         Invitee Name               Invitee Email  \
0         Baani Singh      baanii.singh@gmail.com   
1     Alisa Khvostiuk   alisa.khvostiuk@gmail.com   
2           Ryan Yang        ryan.y2912@gmail.com   
3         Baani Singh      baanii.singh@gmail.com   
4  Mya Charlotte Wong        vecchiamya@gmail.com   
5              Nathan   nathanyuukiwong@gmail.com   
6          Gloria Gao      gloriagao573@gmail.com   
7    Venetia Adamidis  venetiaadamidis8@gmail.com   
8            Nur Shah      nuralmasshah@gmail.com   
9         Michael Chu    michaelchuc123@gmail.com   

                 Event Type Name   Start Date & Time     End Date & Time  \
0  ON: Volunteer Success Program  2025-03-11 4:45 PM  2025-03-11 5:00 PM   
1  ON: Volunteer Success Program  2025-03-03 3:00 PM  2025-03-03 3:15 PM   
2  ON: Volunteer Success Program  2025-03-03 3:45 PM  2025-03-03 4:00 PM   
3  ON: Volunteer Success Program  2025-03-06 4:15 PM  2025-03-06 4:30 PM   
4  ON: Volunteer Success Program  20

In [44]:
query = """
SELECT *
FROM raw_analytics
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

                         Name      What I Do               Display name  \
0   (Vol. Leader) Victoria H.            NaN  (Vol. Leader) Victoria H.   
1  (Vol. Leader) Zainab Ahmed            NaN                     Zainab   
2               Aadam Lakhani            NaN              Aadam Lakhani   
3               Aadhya Sriram            NaN              Aadhya Sriram   
4                   Aakanksha            NaN                  Aakanksha   
5              Aakash Parwani            NaN             Aakash Parwani   
6                  Aali Vaqar      Volunteer                 Aali Vaqar   
7               Aanchal Ratha            NaN              Aanchal Ratha   
8          Aanushan Elangoban  Youth Advisor         Aanushan Elangoban   
9          Aanushan Elangoban            NaN         Aanushan Elangoban   

                                 Email Account type  Messages posted  \
0                 iivv.berry@gmail.com       Member                0   
1        zainab.ahmed@yorkeduc

## Clean datasets

In [61]:
con.execute(r"""
CREATE OR REPLACE VIEW clean_volunteers AS
WITH base AS (
  SELECT
    TRIM(Name) AS full_name,

    -- 1) extract "Sep 26, 2025 3:07 AM" from the messy text
    regexp_extract(
      "Creation Log",
      '([A-Z][a-z]{2}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2}\s+[AP]M)',
      1
    ) AS date_joined_vps,

    LOWER(TRIM("Email Address")) AS email,
    LOWER(TRIM("YRES Email")) AS yres_email,
    TRIM("Youth Advisor") AS youth_advisor

  FROM raw_volunteers
)
SELECT
  full_name,

  -- 2) parse into TIMESTAMP (safe)
  TRY_STRPTIME(date_joined_vps, '%b %d, %Y %I:%M %p') AS date_joined_vps,

  email,
  yres_email,
  youth_advisor
FROM base
WHERE full_name IS NOT NULL
""")

df_volunteers_clean = con.execute("""
SELECT *
FROM clean_volunteers
LIMIT 20
""").df()

df_volunteers_clean


,full_name,date_joined_vps,email,yres_email,youth_advisor
0,Mariam Ali,2025-09-11 13:30:00,mariam.ali@yorkeducation.ca,NaN,Bella He
1,Manusha Srikanthan,2025-09-11 13:45:00,manushasrikanthan@gmail.com,NaN,NaN
2,Harvi Shah,2025-09-11 16:36:00,harvishah2602@gmail.com,harvi.shah@yorkeducation.ca,Tiffany Ye
3,Dorsey Bangarh,2025-09-11 17:37:00,dorseybangarh2004@gmail.com,dorsey.bangarh@yorkeducation.ca,NaN
4,Pushti Ladani,2025-09-11 19:26:00,pushti-kantilal.ladani@mohawkcollege.ca,pushti.ladani@yorkeducation.ca,NaN
5,Purti Ladani,2025-09-11 19:29:00,purti1002@gmail.com,purti.ladani@yorkeducation.ca,NaN
6,Harjot Singh,2025-09-11 20:52:00,singhharjot1312@gmail.com,harjot.singh@yorkeducation.ca,Aanushan Elangoban
7,Rincy Nahomie,2025-09-11 20:31:00,rincynahomie85@gmail.com,rincy.nahomie@yorkeducation.ca,NaN
8,JeeHu Choi,2025-09-11 20:55:00,jiwhotwin@gmail.com,jeehu.choi@yorkeducation.ca,Bella He
9,Taranveer Arora,2025-09-11 21:32:00,taransarora@gmail.com,taranveer.arora@yorkeducation.ca,Bryna McGarrigle


In [51]:
con.execute("""
CREATE OR REPLACE VIEW clean_interviews AS
WITH base AS (
    SELECT
        TRIM("Invitee Name") AS full_name,
        LOWER(TRIM("Invitee Email")) AS email,
        LOWER(TRIM("Event Type Name")) AS event_type,
        LOWER("Start Date & Time") AS event_start_str,
        LOWER("End Date & Time") AS event_end_str,
        "Canceled" AS canceled_flag,
        LOWER(TRIM("Canceled By")) AS canceled_by,
        

    FROM raw_interviews
)
SELECT *
FROM base
""")
df_interviews_clean = con.execute("""
SELECT *
FROM clean_interviews
LIMIT 20
""").df()
df_interviews_clean

,full_name,email,event_type,event_start_str,event_end_str,canceled_flag,canceled_by
0,Baani Singh,baanii.singh@gmail.com,on: volunteer success program,2025-03-11 4:45 pm,2025-03-11 5:00 pm,True,host
1,Alisa Khvostiuk,alisa.khvostiuk@gmail.com,on: volunteer success program,2025-03-03 3:00 pm,2025-03-03 3:15 pm,False,NaN
2,Ryan Yang,ryan.y2912@gmail.com,on: volunteer success program,2025-03-03 3:45 pm,2025-03-03 4:00 pm,False,NaN
3,Baani Singh,baanii.singh@gmail.com,on: volunteer success program,2025-03-06 4:15 pm,2025-03-06 4:30 pm,False,NaN
4,Mya Charlotte Wong,vecchiamya@gmail.com,on: volunteer success program,2025-03-03 3:30 pm,2025-03-03 3:45 pm,False,NaN
5,Nathan,nathanyuukiwong@gmail.com,on: volunteer success program,2025-03-04 3:30 pm,2025-03-04 3:45 pm,False,NaN
6,Gloria Gao,gloriagao573@gmail.com,on: volunteer success program,2025-03-03 3:15 pm,2025-03-03 3:30 pm,False,NaN
7,Venetia Adamidis,venetiaadamidis8@gmail.com,on: volunteer success program,2025-03-04 3:45 pm,2025-03-04 4:00 pm,False,NaN
8,Nur Shah,nuralmasshah@gmail.com,on: volunteer success program,2025-03-07 4:00 pm,2025-03-07 4:15 pm,True,invitee
9,Michael Chu,michaelchuc123@gmail.com,on: volunteer success program,2025-03-07 4:30 pm,2025-03-07 4:45 pm,True,invitee


In [79]:
con.execute("""
CREATE OR REPLACE VIEW clean_analytics AS
WITH base AS (
    SELECT
        TRIM(Name) AS full_name,

        TRIM("Display name") AS display_name,

        LOWER(TRIM(Email)) AS email,

        TRIM("Account type") AS account_type,

       TRY_STRPTIME("Last active (UTC)", '%b %d, %Y')::DATE AS last_active_date,


        TRY_CAST("Messages posted" AS INTEGER) AS messages_posted,
        TRY_CAST("Messages posted in channels" AS INTEGER) AS messages_posted_in_channels,
        TRY_CAST("Reactions added" AS INTEGER) AS reactions_added

    FROM raw_analytics
)
SELECT *
FROM base
WHERE email IS NOT NULL
""")
df_analytics_clean = con.execute("""
SELECT *
FROM clean_analytics
LIMIT 20
""").df()
df_analytics_clean


,full_name,display_name,email,account_type,last_active_date,messages_posted,messages_posted_in_channels,reactions_added
0,(Vol. Leader) Victoria H.,(Vol. Leader) Victoria H.,iivv.berry@gmail.com,Member,NaT,0,0,0
1,(Vol. Leader) Zainab Ahmed,Zainab,zainab.ahmed@yorkeducation.ca,Member,2025-12-30,0,0,0
2,Aadam Lakhani,Aadam Lakhani,aadam.lakhani@yorkeducation.ca,Member,NaT,0,0,0
3,Aadhya Sriram,Aadhya Sriram,aadhya.sriram@yorkeducation.ca,Member,2024-08-02,0,0,0
4,Aakanksha,Aakanksha,aakanksha.patel@yorkeducation.ca,Member,2025-11-20,0,0,0
5,Aakash Parwani,Aakash Parwani,aakash.parwani@yorkeducation.ca,Member,2026-01-30,3,1,0
6,Aali Vaqar,Aali Vaqar,aali.vaqarahmad@yorkeducation.ca,Member,2025-04-29,0,0,0
7,Aanchal Ratha,Aanchal Ratha,aanchal.ratha@yorkeducation.ca,Member,2024-12-11,0,0,0
8,Aanushan Elangoban,Aanushan Elangoban,aanushan.elangoban@yorkeducation.ca,Admin,2026-02-03,378,27,48
9,Aanushan Elangoban,Aanushan Elangoban,aanushan.236@gmail.com,Admin,2025-09-14,0,0,0


## Task 1. Among all VSP volunteers, who hasn’t booked an interview?

In [54]:
df_analytics_clean.columns, df_interviews_clean.columns, df_volunteers_clean.columns

(Index(['full_name', 'display_name', 'email', 'account_type',
        'last_active_date', 'messages_posted', 'messages_posted_in_channels',
        'reactions_added'],
       dtype='str'),
 Index(['full_name', 'email', 'event_type', 'event_start_str', 'event_end_str',
        'canceled_flag', 'canceled_by'],
       dtype='str'),
 Index(['full_name', 'creation_ts', 'email', 'yres_email', 'youth_advisor'], dtype='str'))

In [69]:
query = """
            SELECT v.full_name, v.email, v.date_joined_vps,
            FROM clean_volunteers v
            LEFT JOIN clean_interviews i
            ON v.email = i.email
            WHERE i.email IS NULL
            ORDER BY v.date_joined_vps DESC
            """
df_no_interviews = con.execute(query).df()
df_no_interviews.to_excel('tables/no_interviews_table.xlsx', index=False)                      

## Question 2. Among all VSP volunteers who have been interviewed, who haven’t been invited to Slack?

In [76]:
query = """
SELECT v.full_name, v.email, v.yres_email, v.date_joined_vps, v.youth_advisor
FROM clean_volunteers v
LEFT JOIN clean_analytics a
ON v.yres_email = a.email
WHERE a.email IS NULL AND v.youth_advisor IS NOT NULL
ORDER BY v.date_joined_vps DESC
"""
df_no_slack = con.execute(query).df()
df_no_slack.to_excel('tables/no_slack_table.xlsx', index=False)

## Question 3. Among all VSP volunteers, who are invited to Slack but never joined?

In [75]:
query = """
SELECT v.full_name, v.email, v.yres_email, v.date_joined_vps, v.youth_advisor, a.account_type
FROM clean_volunteers v
LEFT JOIN clean_analytics a
ON v.yres_email = a.email
WHERE a.account_type = 'Invited Member'
AND v.yres_email NOT IN (
      SELECT email
      FROM clean_analytics
      WHERE account_type = 'Member'
  )
ORDER BY v.date_joined_vps DESC
"""
df_never_joined = con.execute(query).df()
df_never_joined.to_excel('tables/never_joined_table.xlsx', index=False)

## Question 4. Among all VSP volunteers who are on Slack already, but haven’t been active since Nov 30, 2025

In [84]:
query = """
SELECT v.full_name, v.email, v.yres_email, v.date_joined_vps, v.youth_advisor, a.last_active_date
FROM clean_volunteers v
LEFT JOIN clean_analytics a
ON v.yres_email = a.email
WHERE a.last_active_date <= '2025-11-30'
ORDER BY v.date_joined_vps DESC
"""
df_inactive = con.execute(query).df()
df_inactive.to_excel('tables/inactive_volunteers_table.xlsx', index=False)

## Question 5. Slack messages posted by VSP volunteers in January 2026

In [86]:
query = """
SELECT v.full_name, v.email, v.yres_email, v.date_joined_vps, v.youth_advisor, a.last_active_date, a.messages_posted, a.messages_posted_in_channels, a.reactions_added
FROM clean_volunteers v
LEFT JOIN clean_analytics a
ON v.yres_email = a.email
WHERE a.last_active_date >= '2026-01-01' AND a.last_active_date < '2026-02-01'
ORDER BY a.messages_posted DESC
"""
df_january_activity = con.execute(query).df()
df_january_activity.to_excel('tables/january_2026_activity_table.xlsx', index=False)